# Simulated Annealing cho bài toán xếp lịch trực điều dưỡng (Nurse Rostering) trên bộ dữ liệu chuẩn

Notebook thực nghiệm kèm bài báo. Chạy được trên Google Colab hoặc máy cá nhân, chỉ cần CPU.

**Dữ liệu:** Nurse Rostering Benchmark Instances (Curtois & Qu, 2014), Instance1–12, tự tải từ <https://schedulingbenchmarks.org/nrp/>.

**Phương pháp:** Greedy, Hill Climbing (HC), Simulated Annealing (SA), quy hoạch nguyên giải bằng HiGHS (MILP); phần bổ sung R0–R10: HiGHS 60 s, SA 600 s, HC khởi động lại ngẫu nhiên, độ nhạy tham số, phân tích thiếu/thừa người.

**Cách dùng:** xem README.md. Kết quả đã chạy nằm trong thư mục `nrp_sa_experiment/`; mặc định notebook đọc lại kết quả này (`RUN_ORIGINAL = False`) và bỏ qua mọi lần chạy đã có.

## 1. Đọc dữ liệu
Định nghĩa cấu trúc `Instance` và hàm đọc file `.txt` của bộ dữ liệu chuẩn (Curtois & Qu, schedulingbenchmarks.org).

In [ ]:
import os, re, math, time, json, random, zipfile, urllib.request
from dataclasses import dataclass, field
import numpy as np


@dataclass
class Instance:
    """Một instance của bài toán xếp lịch trực điều dưỡng (định dạng schedulingbenchmarks.org).

    Quy ước mã ca: 0 = nghỉ (off), 1..S = các loại ca theo thứ tự trong SECTION_SHIFTS.
    """
    name: str
    D: int                       # số ngày trong kỳ lập lịch (ngày 0 là thứ Hai)
    shift_ids: list              # tên ca, ví dụ ['E', 'L']
    length: list                 # length[s] = số phút của ca s (length[0] = 0)
    forbid: list                 # forbid[a][b] = True nếu ca b KHÔNG được xếp ngay sau ca a
    staff_ids: list
    max_shifts: list             # max_shifts[n][s]
    max_min: list
    min_min: list
    max_cons: list               # số ca liên tiếp tối đa
    min_cons: list               # số ca liên tiếp tối thiểu
    min_off: list                # số ngày nghỉ liên tiếp tối thiểu
    max_wk: list                 # số cuối tuần làm việc tối đa
    days_off: list               # days_off[n] = tập ngày bắt buộc nghỉ
    on_w: np.ndarray             # on_w[n, d, s]  = trọng số yêu cầu ĐƯỢC làm ca s ngày d
    off_w: np.ndarray            # off_w[n, d, s] = trọng số yêu cầu KHÔNG làm ca s ngày d
    cov_req: np.ndarray          # cov_req[d, s]  = số người cần
    cov_under: np.ndarray        # trọng số phạt thiếu người
    cov_over: np.ndarray         # trọng số phạt thừa người

    @property
    def N(self):
        return len(self.staff_ids)

    @property
    def S(self):
        return len(self.shift_ids)


def parse_instance(path):
    """Đọc file .txt của bộ Instance1..24."""
    name = os.path.splitext(os.path.basename(path))[0]
    sections, cur = {}, None
    with open(path, encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("SECTION_"):
                cur = line
                sections[cur] = []
            elif cur is not None:
                sections[cur].append(line)

    D = int(sections["SECTION_HORIZON"][0])

    shift_ids, lens, follow = [], [0], []
    for line in sections["SECTION_SHIFTS"]:
        p = [x.strip() for x in line.split(",")]
        shift_ids.append(p[0])
        lens.append(int(p[1]))
        follow.append([x for x in p[2].split("|") if x] if len(p) > 2 else [])
    S = len(shift_ids)
    sidx = {s: i + 1 for i, s in enumerate(shift_ids)}
    forbid = [[False] * (S + 1) for _ in range(S + 1)]
    for i, lst in enumerate(follow):
        for b in lst:
            forbid[i + 1][sidx[b]] = True

    staff_ids, max_shifts = [], []
    max_min, min_min, max_cons, min_cons, min_off, max_wk = [], [], [], [], [], []
    for line in sections["SECTION_STAFF"]:
        p = [x.strip() for x in line.split(",")]
        staff_ids.append(p[0])
        ms = [0] * (S + 1)
        for item in p[1].split("|"):
            if item:
                k, v = item.split("=")
                ms[sidx[k]] = int(v)
        max_shifts.append(ms)
        max_min.append(int(p[2])); min_min.append(int(p[3]))
        max_cons.append(int(p[4])); min_cons.append(int(p[5]))
        min_off.append(int(p[6])); max_wk.append(int(p[7]))
    N = len(staff_ids)
    nidx = {s: i for i, s in enumerate(staff_ids)}

    days_off = [set() for _ in range(N)]
    for line in sections.get("SECTION_DAYS_OFF", []):
        p = [x.strip() for x in line.split(",")]
        days_off[nidx[p[0]]].update(int(x) for x in p[1:] if x != "")

    on_w = np.zeros((N, D, S + 1), dtype=np.int64)
    off_w = np.zeros((N, D, S + 1), dtype=np.int64)
    for line in sections.get("SECTION_SHIFT_ON_REQUESTS", []):
        e, d, s, w = [x.strip() for x in line.split(",")]
        on_w[nidx[e], int(d), sidx[s]] += int(w)
    for line in sections.get("SECTION_SHIFT_OFF_REQUESTS", []):
        e, d, s, w = [x.strip() for x in line.split(",")]
        off_w[nidx[e], int(d), sidx[s]] += int(w)

    cov_req = np.zeros((D, S + 1), dtype=np.int64)
    cov_under = np.zeros((D, S + 1), dtype=np.int64)
    cov_over = np.zeros((D, S + 1), dtype=np.int64)
    for line in sections["SECTION_COVER"]:
        d, s, r, wu, wo = [x.strip() for x in line.split(",")]
        cov_req[int(d), sidx[s]] = int(r)
        cov_under[int(d), sidx[s]] = int(wu)
        cov_over[int(d), sidx[s]] = int(wo)

    return Instance(name, D, shift_ids, lens, forbid, staff_ids, max_shifts, max_min, min_min,
                    max_cons, min_cons, min_off, max_wk, days_off, on_w, off_w,
                    cov_req, cov_under, cov_over)

## 2. Bộ kiểm tra lời giải độc lập
`evaluate()` kiểm tra đủ 9 ràng buộc cứng và tính điểm phạt mềm theo đúng đặc tả của benchmark. Mọi kết quả báo cáo đều đi qua hàm này.

In [ ]:
def evaluate(inst, roster):
    """Bộ kiểm tra ĐỘC LẬP (viết lại từ đặc tả, không dùng chung mã với SA).

    roster[n][d] là mã ca (0 = nghỉ). Trả về dict gồm:
      - 'hard': tổng số vi phạm ràng buộc cứng (0 nghĩa là lời giải khả thi)
      - 'soft': tổng điểm phạt (hàm mục tiêu của benchmark)
      - 'detail': số vi phạm theo từng loại ràng buộc
    """
    N, D, S = inst.N, inst.D, inst.S
    R = np.asarray(roster, dtype=np.int64)
    assert R.shape == (N, D), "Kích thước lịch không khớp instance"
    det = dict(days_off=0, rotation=0, max_shifts=0, max_minutes=0, min_minutes=0,
               max_consecutive=0, min_consecutive=0, min_days_off=0, max_weekends=0)
    soft_req = 0
    for n in range(N):
        row = R[n]
        work = row > 0
        det["days_off"] += sum(1 for d in inst.days_off[n] if work[d])
        for d in range(D - 1):
            if inst.forbid[row[d]][row[d + 1]]:
                det["rotation"] += 1
        for s in range(1, S + 1):
            c = int((row == s).sum())
            if c > inst.max_shifts[n][s]:
                det["max_shifts"] += 1
        minutes = sum(inst.length[v] for v in row)
        if minutes > inst.max_min[n]:
            det["max_minutes"] += 1
        if minutes < inst.min_min[n]:
            det["min_minutes"] += 1
        # tách thành các khối liên tiếp (giá trị, ngày bắt đầu, độ dài)
        blocks, st = [], 0
        for d in range(1, D + 1):
            if d == D or work[d] != work[st]:
                blocks.append((bool(work[st]), st, d - st))
                st = d
        for is_work, st, ln in blocks:
            touches_edge = (st == 0) or (st + ln == D)
            if is_work:
                if ln > inst.max_cons[n]:
                    det["max_consecutive"] += 1
                # biên kỳ lập lịch: coi như có vô hạn ca làm ở kỳ trước/sau -> không phạt
                if ln < inst.min_cons[n] and not touches_edge:
                    det["min_consecutive"] += 1
            else:
                if ln < inst.min_off[n] and not touches_edge:
                    det["min_days_off"] += 1
        wk = sum(1 for w in range(D // 7) if work[7 * w + 5] or work[7 * w + 6])
        if wk > inst.max_wk[n]:
            det["max_weekends"] += 1
        for d in range(D):
            v = row[d]
            for s in range(1, S + 1):
                if s != v:
                    soft_req += int(inst.on_w[n, d, s])     # muốn làm ca s nhưng không được xếp
            if v > 0:
                soft_req += int(inst.off_w[n, d, v])        # không muốn làm ca v nhưng bị xếp
    soft_cov = 0
    for d in range(D):
        for s in range(1, S + 1):
            x = int((R[:, d] == s).sum())
            r = int(inst.cov_req[d, s])
            if x < r:
                soft_cov += (r - x) * int(inst.cov_under[d, s])
            elif x > r:
                soft_cov += (x - r) * int(inst.cov_over[d, s])
    hard = sum(det.values())
    return dict(hard=hard, soft=soft_req + soft_cov, soft_requests=soft_req,
                soft_cover=soft_cov, feasible=(hard == 0), detail=det)


def roster_to_text(inst, roster):
    """Xuất lời giải theo định dạng tab của RosterViewer (mỗi dòng một nhân viên, ô trống = nghỉ)."""
    names = [""] + inst.shift_ids
    return "\n".join("\t".join(names[v] for v in row) for row in roster)

## 3. Đánh giá gia tăng cho SA
Mỗi nước đi chỉ ảnh hưởng 1–2 nhân viên, nên chỉ cần tính lại chi phí của các nhân viên đó.

In [ ]:
def build_params(inst):
    """Tiền xử lý tham số theo từng nhân viên để đánh giá gia tăng (delta) nhanh trong SA."""
    N, D, S = inst.N, inst.D, inst.S
    unit = min(inst.length[1:])
    # rc[n][d][v] = điểm phạt yêu cầu cá nhân nếu nhân viên n nhận giá trị v vào ngày d
    rc_all = []
    for n in range(N):
        tot_on = inst.on_w[n].sum(axis=1)                       # (D,)
        rc = (tot_on[:, None] - inst.on_w[n] + inst.off_w[n])   # (D, S+1)
        rc[:, 0] = tot_on
        rc_all.append(rc.tolist())
    P = []
    for n in range(N):
        P.append((D, S, inst.length, inst.forbid, inst.max_shifts[n], inst.max_min[n],
                  inst.min_min[n], inst.max_cons[n], inst.min_cons[n], inst.min_off[n],
                  inst.max_wk[n], rc_all[n], unit, D // 7))
    allowed = [[0] + [s for s in range(1, S + 1) if inst.max_shifts[n][s] > 0] for n in range(N)]
    free_days = [[d for d in range(D) if d not in inst.days_off[n]] for n in range(N)]
    return P, allowed, free_days


def emp_eval(row, p):
    """Trả về (mức vi phạm cứng, điểm phạt mềm) của MỘT nhân viên.

    Mức vi phạm cứng được đo theo 'độ lớn' (ví dụ làm 7 ca liên tiếp khi tối đa 5 -> 2)
    để SA có gradient hướng về miền khả thi.
    """
    D, S, lens, forb, maxs, maxm, minm, maxc, minc, mino, maxw, rc, unit, W = p
    hard = 0
    soft = 0
    minutes = 0
    cnt = [0] * (S + 1)
    prev = 0
    bs = 0          # ngày bắt đầu của khối hiện tại
    for d in range(D):
        v = row[d]
        soft += rc[d][v]
        if v:
            minutes += lens[v]
            cnt[v] += 1
            if prev:
                if forb[prev][v]:
                    hard += 1
            else:
                ln = d - bs                      # kết thúc một khối nghỉ
                if bs > 0 and ln < mino:
                    hard += mino - ln
                bs = d
        elif prev:
            ln = d - bs                          # kết thúc một khối làm việc
            if ln > maxc:
                hard += ln - maxc
            if bs > 0 and ln < minc:
                hard += minc - ln
            bs = d
        prev = v
    if prev:
        ln = D - bs
        if ln > maxc:
            hard += ln - maxc
    for s in range(1, S + 1):
        if cnt[s] > maxs[s]:
            hard += cnt[s] - maxs[s]
    if minutes > maxm:
        hard += -(-(minutes - maxm) // unit)
    elif minutes < minm:
        hard += -(-(minm - minutes) // unit)
    wk = 0
    for w in range(W):
        if row[7 * w + 5] or row[7 * w + 6]:
            wk += 1
    if wk > maxw:
        hard += wk - maxw
    return hard, soft

## 4. Heuristic tham lam (baseline 1 và lời giải khởi tạo)

In [ ]:
def greedy_construct(inst, rng=None):
    """Heuristic tham lam: duyệt từng ngày, từng ca, xếp đủ số người yêu cầu nếu còn ứng viên hợp lệ.

    Dùng làm (1) baseline đơn giản và (2) lời giải khởi tạo cho Hill Climbing / SA.
    Heuristic này bảo đảm: ngày nghỉ bắt buộc, luân phiên ca, số ca tối đa, số phút tối đa,
    số ca liên tiếp tối đa, số cuối tuần tối đa. Các ràng buộc 'tối thiểu' có thể còn vi phạm.
    """
    rng = rng or random.Random(0)
    N, D, S = inst.N, inst.D, inst.S
    roster = [[0] * D for _ in range(N)]
    minutes = [0] * N
    cnt = [[0] * (S + 1) for _ in range(N)]
    run = [0] * N          # số ca liên tiếp tính đến hôm qua
    offrun = [99] * N      # số ngày nghỉ liên tiếp tính đến hôm qua
    wk_used = [0] * N
    for d in range(D):
        is_we, w = (d % 7) >= 5, d // 7
        sat_worked = [(d % 7 == 6) and roster[n][d - 1] > 0 for n in range(N)]
        order = sorted(range(1, S + 1), key=lambda s: -int(inst.cov_req[d, s] * inst.cov_under[d, s]))
        for s in order:
            need = int(inst.cov_req[d, s])
            cands = []
            for n in range(N):
                if roster[n][d] or d in inst.days_off[n]:
                    continue
                if cnt[n][s] >= inst.max_shifts[n][s]:
                    continue
                if minutes[n] + inst.length[s] > inst.max_min[n]:
                    continue
                if d > 0 and inst.forbid[roster[n][d - 1]][s]:
                    continue
                if run[n] >= inst.max_cons[n]:
                    continue
                if is_we and not sat_worked[n] and wk_used[n] >= inst.max_wk[n]:
                    continue
                # ưu tiên: đang dở khối làm việc chưa đủ dài > đã nghỉ đủ > còn thiếu nhiều phút
                continuing = 0 < run[n] < inst.min_cons[n]
                rested = run[n] > 0 or offrun[n] >= inst.min_off[n]
                deficit = inst.min_min[n] - minutes[n]
                pref = int(inst.on_w[n, d, s]) - int(inst.off_w[n, d, s])
                cands.append((-int(continuing), -int(rested), -pref, -deficit, rng.random(), n))
            cands.sort()
            for c in cands[:need]:
                n = c[-1]
                roster[n][d] = s
                minutes[n] += inst.length[s]
                cnt[n][s] += 1
                if is_we and not sat_worked[n]:
                    wk_used[n] += 1
        for n in range(N):
            if roster[n][d]:
                run[n] += 1; offrun[n] = 0
            else:
                run[n] = 0; offrun[n] += 1
    return roster

## 5. Simulated Annealing và Hill Climbing
Hill Climbing dùng cùng lân cận nhưng chỉ nhận nước đi không làm xấu lời giải (baseline 2), giúp tách riêng đóng góp của cơ chế chấp nhận trong SA.

In [ ]:
def local_search(inst, time_limit, seed=0, method="SA", T0=None, Tf=0.3, hard_weight=None,
                 p_moves=(0.35, 0.35, 0.30), max_block=4, init=None, reheat=False):
    """Simulated Annealing (method='SA') hoặc Hill Climbing (method='HC') cho bài toán NRP.

    Hàm chi phí:  F = soft + hard_weight * hard
    Ba phép biến đổi lân cận:
      1) CHANGE   : gán một ca (hoặc nghỉ) cho khối 1..max_block ngày liên tiếp của một nhân viên
      2) SWAP-EMP : hoán đổi một khối 1..max_block ngày liên tiếp giữa hai nhân viên
      3) SWAP-DAY : hoán đổi ca giữa hai ngày của cùng một nhân viên
    Làm nguội hình học theo THỜI GIAN: T(t) = T0 * (Tf/T0)^(t/time_limit), nên thuật toán luôn
    dùng hết đúng ngân sách thời gian. Khi reheat=True, mỗi chu kỳ làm nguội chiếm 1/4 thời gian
    rồi hâm nóng lại từ lời giải tốt nhất (4 chu kỳ).

    Mặc định (quy tắc theo thang trọng số của instance, w_max = trọng số phạt thiếu người lớn nhất):
      T0 = w_max            -> ở nhiệt độ đầu, nước đi làm thiếu 1 người vẫn được nhận với xác suất ~ e^-1
      hard_weight = 2*w_max -> vi phạm ràng buộc cứng luôn đắt hơn mọi vi phạm mềm đơn lẻ
    """
    w_max = float(max(1, int(inst.cov_under.max())))
    if T0 is None:
        T0 = w_max
    if hard_weight is None:
        hard_weight = 2 * w_max
    rng = random.Random(seed)
    rnd, rint = rng.random, rng.randrange
    N, D, S = inst.N, inst.D, inst.S
    P, allowed, free_days = build_params(inst)
    dayoff = [[(d in inst.days_off[n]) for d in range(D)] for n in range(N)]
    can = [[inst.max_shifts[n][s] > 0 for s in range(S + 1)] for n in range(N)]
    for n in range(N):
        can[n][0] = True
    req = inst.cov_req.tolist(); wu = inst.cov_under.tolist(); wo = inst.cov_over.tolist()

    roster = [list(r) for r in (init if init is not None else greedy_construct(inst, rng))]
    cnt = [[0] * (S + 1) for _ in range(D)]
    for n in range(N):
        for d in range(D):
            cnt[d][roster[n][d]] += 1
    eh, es = [0] * N, [0] * N
    for n in range(N):
        eh[n], es[n] = emp_eval(roster[n], P[n])
    cov = 0
    for d in range(D):
        for s in range(1, S + 1):
            x, r = cnt[d][s], req[d][s]
            cov += (r - x) * wu[d][s] if x < r else (x - r) * wo[d][s]
    hard, soft = sum(eh), sum(es) + cov

    def cov_delta(d, a, b):
        """Thay đổi điểm phạt cover khi một ô ngày d đổi từ ca a sang ca b."""
        dl = 0
        if a:
            x, r = cnt[d][a], req[d][a]
            dl += (wu[d][a] if x <= r else -wo[d][a])
        if b:
            x, r = cnt[d][b], req[d][b]
            dl += (-wu[d][b] if x < r else wo[d][b])
        return dl

    HW = hard_weight
    best_feas, best_feas_roster = None, None      # lời giải khả thi tốt nhất
    best_F, best_F_roster = soft + HW * hard, [r[:] for r in roster]
    if hard == 0:
        best_feas, best_feas_roster = soft, [r[:] for r in roster]
    trace = [(0.0, best_feas if best_feas is not None else float("nan"))]
    t_best = 0.0
    is_sa = (method.upper() == "SA")
    n_cycles = 4 if (is_sa and reheat) else 1
    cycle_len = time_limit / n_cycles
    ratio = (Tf / T0) if is_sa else 1.0
    T = T0 if is_sa else 0.0
    p1, p2 = p_moves[0], p_moves[0] + p_moves[1]
    it = acc = 0
    cycle = 0
    t0 = time.perf_counter()
    exp = math.exp
    while True:
        if (it & 255) == 0:
            el = time.perf_counter() - t0
            if el >= time_limit:
                break
            if is_sa:
                c = int(el / cycle_len)
                if c != cycle:                     # hâm nóng: quay về lời giải tốt nhất
                    cycle = c
                    src = best_feas_roster if best_feas_roster is not None else best_F_roster
                    roster = [r[:] for r in src]
                    cnt = [[0] * (S + 1) for _ in range(D)]
                    for n in range(N):
                        for d in range(D):
                            cnt[d][roster[n][d]] += 1
                        eh[n], es[n] = emp_eval(roster[n], P[n])
                    cov = 0
                    for d in range(D):
                        for s in range(1, S + 1):
                            x, r = cnt[d][s], req[d][s]
                            cov += (r - x) * wu[d][s] if x < r else (x - r) * wo[d][s]
                    hard, soft = sum(eh), sum(es) + cov
                T = T0 * ratio ** ((el - cycle * cycle_len) / cycle_len)
        it += 1
        u = rnd()
        if u < p1:                                  # ---- CHANGE (khối 1..max_block ngày của 1 nhân viên)
            n = rint(N)
            al = allowed[n]
            b = al[rint(len(al))]
            k = 1 if rnd() < 0.5 else 1 + rint(max_block)
            d0 = rint(D - k + 1)
            row = roster[n]
            dof = dayoff[n]
            ch = [(d, row[d]) for d in range(d0, d0 + k) if row[d] != b and not dof[d]]
            if not ch:
                continue
            dc = 0
            for d, a in ch:
                dc += cov_delta(d, a, b)
                cnt[d][a] -= 1; cnt[d][b] += 1
                row[d] = b
            h1, s1 = emp_eval(row, P[n])
            dh = h1 - eh[n]
            ds = s1 - es[n] + dc
            delta = ds + HW * dh
            if delta <= 0 or (T > 0 and rnd() < exp(-delta / T)):
                eh[n], es[n] = h1, s1
                hard += dh; soft += ds; acc += 1
            else:
                for d, a in ch:
                    cnt[d][a] += 1; cnt[d][b] -= 1
                    row[d] = a
                continue
        elif u < p2:                                # ---- SWAP-EMP (khối ngày giữa 2 nhân viên)
            n1 = rint(N); n2 = rint(N)
            if n1 == n2:
                continue
            k = 1 + rint(max_block)
            d0 = rint(D - k + 1)
            r1, r2 = roster[n1], roster[n2]
            ok = False
            for d in range(d0, d0 + k):
                a, b = r1[d], r2[d]
                if a != b:
                    ok = True
                    if (b and (dayoff[n1][d] or not can[n1][b])) or (a and (dayoff[n2][d] or not can[n2][a])):
                        ok = False
                        break
            if not ok:
                continue
            seg1, seg2 = r1[d0:d0 + k], r2[d0:d0 + k]
            r1[d0:d0 + k] = seg2; r2[d0:d0 + k] = seg1
            h1, s1 = emp_eval(r1, P[n1]); h2, s2 = emp_eval(r2, P[n2])
            dh = h1 + h2 - eh[n1] - eh[n2]
            ds = s1 + s2 - es[n1] - es[n2]          # cover không đổi
            delta = ds + HW * dh
            if delta <= 0 or (T > 0 and rnd() < exp(-delta / T)):
                eh[n1], es[n1], eh[n2], es[n2] = h1, s1, h2, s2
                hard += dh; soft += ds; acc += 1
            else:
                r1[d0:d0 + k] = seg1; r2[d0:d0 + k] = seg2
                continue
        else:                                       # ---- SWAP-DAY (2 ngày của 1 nhân viên)
            n = rint(N)
            fd = free_days[n]
            if len(fd) < 2:
                continue
            d1 = fd[rint(len(fd))]; d2 = fd[rint(len(fd))]
            row = roster[n]
            a, b = row[d1], row[d2]
            if a == b:
                continue
            row[d1], row[d2] = b, a
            h1, s1 = emp_eval(row, P[n])
            dc = cov_delta(d1, a, b)
            cnt[d1][a] -= 1; cnt[d1][b] += 1        # cập nhật tạm để tính ngày thứ hai
            dc += cov_delta(d2, b, a)
            dh = h1 - eh[n]
            ds = s1 - es[n] + dc
            delta = ds + HW * dh
            if delta <= 0 or (T > 0 and rnd() < exp(-delta / T)):
                eh[n], es[n] = h1, s1
                cnt[d2][b] -= 1; cnt[d2][a] += 1
                hard += dh; soft += ds; acc += 1
            else:
                cnt[d1][a] += 1; cnt[d1][b] -= 1
                row[d1], row[d2] = a, b
                continue
        # ---- cập nhật lời giải tốt nhất (chỉ chạy khi nước đi được chấp nhận)
        if hard == 0:
            if best_feas is None or soft < best_feas:
                best_feas = soft
                best_feas_roster = [r[:] for r in roster]
                t_best = time.perf_counter() - t0
                trace.append((t_best, best_feas))
        else:
            F = soft + HW * hard
            if best_feas is None and F < best_F:
                best_F, best_F_roster = F, [r[:] for r in roster]

    elapsed = time.perf_counter() - t0
    final = best_feas_roster if best_feas_roster is not None else best_F_roster
    return dict(roster=final, feasible=best_feas is not None,
                cost=best_feas if best_feas is not None else float("nan"),
                time_to_best=t_best, iterations=it, accepted=acc, elapsed=elapsed, trace=trace)

## 6. Baseline chính xác: mô hình quy hoạch nguyên giải bằng HiGHS (baseline 3)

In [ ]:
def solve_milp(inst, time_limit=300, verbose=False):
    """Baseline chính xác: mô hình quy hoạch nguyên (theo Curtois & Qu) giải bằng HiGHS qua scipy."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    from scipy.sparse import coo_matrix
    N, D, S = inst.N, inst.D, inst.S
    nx = N * D * S
    W = D // 7
    X = lambda n, d, s: (n * D + d) * S + (s - 1)          # s = 1..S
    WK = lambda n, w: nx + n * W + w
    UN = lambda d, s: nx + N * W + d * S + (s - 1)
    OV = lambda d, s: nx + N * W + D * S + d * S + (s - 1)
    nv = nx + N * W + 2 * D * S

    rows, cols, vals, lo, hi = [], [], [], [], []
    r = 0

    def add(terms, lb, ub):
        nonlocal r
        for c, v in terms:
            rows.append(r); cols.append(c); vals.append(v)
        lo.append(lb); hi.append(ub); r += 1

    work = lambda n, d: [(X(n, d, s), 1.0) for s in range(1, S + 1)]
    INF = np.inf
    for n in range(N):
        for d in range(D):
            add(work(n, d), -INF, 1)                                   # tối đa 1 ca/ngày
        for d in range(D - 1):                                         # luân phiên ca
            for a in range(1, S + 1):
                for b in range(1, S + 1):
                    if inst.forbid[a][b]:
                        add([(X(n, d, a), 1.0), (X(n, d + 1, b), 1.0)], -INF, 1)
        for s in range(1, S + 1):                                      # số ca tối đa theo loại
            add([(X(n, d, s), 1.0) for d in range(D)], -INF, inst.max_shifts[n][s])
        add([(X(n, d, s), float(inst.length[s])) for d in range(D) for s in range(1, S + 1)],
            inst.min_min[n], inst.max_min[n])                          # tổng số phút
        mc = inst.max_cons[n]
        for d in range(D - mc):                                        # số ca liên tiếp tối đa
            add([t for j in range(d, d + mc + 1) for t in work(n, j)], -INF, mc)
        for c in range(1, inst.min_cons[n]):                           # số ca liên tiếp tối thiểu
            for d in range(D - c - 1):
                t = [(i, -1.0) for i, _ in work(n, d)] + [(i, -1.0) for i, _ in work(n, d + c + 1)]
                t += [x for j in range(d + 1, d + c + 1) for x in work(n, j)]
                add(t, -INF, c - 1)
        for c in range(1, inst.min_off[n]):                            # số ngày nghỉ liên tiếp tối thiểu
            for d in range(D - c - 1):
                t = work(n, d) + work(n, d + c + 1)
                t += [(i, -1.0) for j in range(d + 1, d + c + 1) for i, _ in work(n, j)]
                add(t, -INF, 1)
        for w in range(W):                                             # cuối tuần
            for d in (7 * w + 5, 7 * w + 6):
                add(work(n, d) + [(WK(n, w), -1.0)], -INF, 0)
        if W:
            add([(WK(n, w), 1.0) for w in range(W)], -INF, inst.max_wk[n])
    for d in range(D):                                                 # cover
        for s in range(1, S + 1):
            add([(X(n, d, s), 1.0) for n in range(N)] + [(UN(d, s), 1.0), (OV(d, s), -1.0)],
                int(inst.cov_req[d, s]), int(inst.cov_req[d, s]))

    c = np.zeros(nv)
    const = 0.0
    lb = np.zeros(nv); ub = np.ones(nv)
    ub[nx + N * W:] = N
    for n in range(N):
        for d in range(D):
            for s in range(1, S + 1):
                c[X(n, d, s)] = float(inst.off_w[n, d, s] - inst.on_w[n, d, s])
                const += float(inst.on_w[n, d, s])
                if d in inst.days_off[n]:
                    ub[X(n, d, s)] = 0
    for d in range(D):
        for s in range(1, S + 1):
            c[UN(d, s)] = float(inst.cov_under[d, s]); c[OV(d, s)] = float(inst.cov_over[d, s])

    A = coo_matrix((vals, (rows, cols)), shape=(r, nv)).tocsr()
    t0 = time.perf_counter()
    res = milp(c, constraints=LinearConstraint(A, np.array(lo, float), np.array(hi, float)),
               integrality=np.ones(nv), bounds=Bounds(lb, ub),
               options=dict(time_limit=float(time_limit), disp=verbose))
    elapsed = time.perf_counter() - t0
    out = dict(elapsed=elapsed, status=int(res.status), message=str(res.message), feasible=False,
               cost=float("nan"), roster=None, proven_optimal=(res.status == 0),
               lower_bound=float("nan"))
    lbnd = getattr(res, "mip_dual_bound", None)
    if lbnd is not None and np.isfinite(lbnd):
        out["lower_bound"] = float(lbnd) + const
    if res.x is not None:
        x = np.rint(res.x[:nx]).astype(int).reshape(N, D, S)
        roster = (x * np.arange(1, S + 1)[None, None, :]).sum(axis=2).tolist()
        ev = evaluate(inst, roster)
        out.update(roster=roster, feasible=ev["feasible"], cost=float(ev["soft"]))
    return out

## 7. Cấu hình thực nghiệm
Đổi `MODE = "paper"` khi đã chạy thử xong. Bật `USE_DRIVE = True` để kết quả được lưu trên Google Drive và tự chạy tiếp nếu Colab ngắt phiên.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------------------------
# CHỌN CHẾ ĐỘ CHẠY
#   "quick": chạy thử ~10 phút để kiểm tra mọi thứ hoạt động
#   "paper": cấu hình đề xuất cho bài báo (xem ước lượng thời gian in ra bên dưới)
# ----------------------------------------------------------------------------------------------
MODE = "paper"

PRESETS = {
    "quick": dict(instances=[1, 2, 3, 4], time_limit=20, n_runs=3, milp_time=60),
    "paper": dict(instances=list(range(1, 13)), time_limit=60, n_runs=10, milp_time=600),
}
CFG = PRESETS[MODE]
INSTANCES  = CFG["instances"]     # chỉ số instance (1..24)
TIME_LIMIT = CFG["time_limit"]    # giây cho MỖI lần chạy SA / HC
N_RUNS     = CFG["n_runs"]        # số lần chạy độc lập (seed = 0..N_RUNS-1)
MILP_TIME  = CFG["milp_time"]     # giây cho baseline MILP (HiGHS)
SA_PARAMS  = dict(T0=None, Tf=0.3, hard_weight=None, p_moves=(0.35, 0.35, 0.30), max_block=4)

USE_DRIVE = False                 # True: lưu kết quả lên Google Drive để chạy tiếp khi Colab ngắt
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/nrp_sa_experiment"
else:
    OUT_DIR = "nrp_sa_experiment"
DATA_DIR = os.path.join(OUT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "solutions"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)

# Lời giải tốt nhất đã biết (Best Known Solution) lấy từ https://schedulingbenchmarks.org/nrp/
# (truy cập 07/10/2026). Instance 1-23 đã được chứng minh tối ưu. Hãy đối chiếu lại trước khi nộp bài.
BKS = {1: 607, 2: 828, 3: 1001, 4: 1716, 5: 1143, 6: 1950, 7: 1056, 8: 1300, 9: 439, 10: 4631,
       11: 3443, 12: 4040, 13: 1348, 14: 1278, 15: 3828, 16: 3225, 17: 5746, 18: 4459, 19: 3149,
       20: 4769, 21: 21133, 22: 30241, 23: 17285, 24: 40439}

_total = len(INSTANCES) * (2 * N_RUNS * TIME_LIMIT + MILP_TIME)
print(f"Chế độ: {MODE} | instance: {INSTANCES} | {N_RUNS} lần chạy x {TIME_LIMIT}s | MILP {MILP_TIME}s")
print(f"Ước lượng tổng thời gian (tối đa): {_total/60:.0f} phút (~{_total/3600:.1f} giờ)")

## 8. Tải bộ dữ liệu từ trang gốc

In [ ]:
BASE_URL = "https://schedulingbenchmarks.org/nrp/data/"

def _fetch(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r, open(dest, "wb") as f:
        f.write(r.read())

def download_instances(indices, data_dir):
    """Tải bộ dữ liệu từ trang gốc. Thử file zip trước, nếu lỗi thì tải từng file .txt."""
    paths = {i: os.path.join(data_dir, f"Instance{i}.txt") for i in indices}
    if all(os.path.exists(p) for p in paths.values()):
        print("Dữ liệu đã có sẵn, bỏ qua bước tải.")
        return paths
    try:
        zpath = os.path.join(data_dir, "instances1_24.zip")
        _fetch(BASE_URL + "instances1_24.zip", zpath)
        with zipfile.ZipFile(zpath) as z:
            for member in z.namelist():
                m = re.fullmatch(r"Instance(\d+)\.txt", os.path.basename(member))
                if m:
                    with z.open(member) as src, open(os.path.join(data_dir, os.path.basename(member)), "wb") as dst:
                        dst.write(src.read())
        print("Đã tải và giải nén instances1_24.zip")
    except Exception as e:
        print("Không tải được file zip (", e, ") -> tải từng file .txt")
    for i, p in paths.items():
        if not os.path.exists(p):
            _fetch(BASE_URL + f"Instance{i}.txt", p)
            print("Đã tải", os.path.basename(p))
    return paths

PATHS = download_instances(INSTANCES, DATA_DIR)
INST = {i: parse_instance(PATHS[i]) for i in INSTANCES}

overview = pd.DataFrame([dict(Instance=i, Weeks=INST[i].D // 7, Days=INST[i].D, Employees=INST[i].N,
                              ShiftTypes=INST[i].S, BKS=BKS[i]) for i in INSTANCES])
overview.to_csv(os.path.join(OUT_DIR, "table_instances.csv"), index=False)
overview

## 9. Kiểm tra tính đúng đắn trước khi chạy

In [ ]:
# KIỂM TRA TÍNH ĐÚNG ĐẮN trước khi chạy thực nghiệm:
#  (a) hàm đánh giá nhanh của SA phải khớp với bộ kiểm tra độc lập evaluate()
#  (b) MILP trên instance nhỏ nhất phải cho đúng giá trị tối ưu đã công bố
def sanity_check(inst, n_trials=200, seed=123):
    rng = random.Random(seed)
    P, allowed, free_days = build_params(inst)
    for _ in range(n_trials):
        ro = [[0] * inst.D for _ in range(inst.N)]
        for n in range(inst.N):
            for d in free_days[n]:
                if rng.random() < 0.6:
                    ro[n][d] = rng.choice(allowed[n])
        ev = evaluate(inst, ro)
        h = s = 0
        for n in range(inst.N):
            a, b = emp_eval(ro[n], P[n]); h += a; s += b
        assert s == ev["soft_requests"], "Lệch điểm phạt yêu cầu cá nhân"
        assert (h == 0) == (ev["hard"] == 0), "Lệch kết luận khả thi"
    return True

for i in INSTANCES:
    sanity_check(INST[i])
print("(a) Hàm đánh giá nhanh khớp với evaluate() trên", len(INSTANCES), "instance.")

_i0 = min(INSTANCES)
_m = solve_milp(INST[_i0], time_limit=120)
print(f"(b) MILP trên Instance{_i0}: cost = {_m['cost']:.0f} | BKS = {BKS[_i0]} | "
      f"chứng minh tối ưu = {_m['proven_optimal']} | {_m['elapsed']:.1f}s")
if _m["proven_optimal"] and abs(_m["cost"] - BKS[_i0]) > 1e-6:
    print("  CẢNH BÁO: MILP tối ưu nhưng khác BKS -> kiểm tra lại dữ liệu / bảng BKS.")

## 10. Chạy thực nghiệm
Kết quả được ghi vào `results_raw.csv` sau từng lần chạy. Chạy lại cell này sẽ bỏ qua các lần đã xong.

In [ ]:
RESULT_CSV = os.path.join(OUT_DIR, "results_raw.csv")
TRACE_JSON = os.path.join(OUT_DIR, "traces.json")
COLS = ["instance", "method", "seed", "time_limit", "cost", "feasible", "hard_violations",
        "time_to_best", "iterations", "elapsed", "proven_optimal", "lower_bound"]

def load_results():
    if os.path.exists(RESULT_CSV):
        return pd.read_csv(RESULT_CSV)
    return pd.DataFrame(columns=COLS)

def load_traces():
    if os.path.exists(TRACE_JSON):
        with open(TRACE_JSON) as f:
            return json.load(f)
    return {}

def already_done(df, i, method, seed, tl):
    if df.empty:
        return False
    m = (df.instance == i) & (df.method == method) & (df.seed == seed) & (df.time_limit == tl)
    return bool(m.any())

def append_row(row):
    df = pd.DataFrame([{c: row.get(c, np.nan) for c in COLS}])
    df.to_csv(RESULT_CSV, mode="a", header=not os.path.exists(RESULT_CSV), index=False)

def save_solution(i, method, inst, roster, cost):
    """Lưu lời giải tốt nhất của mỗi (instance, phương pháp) theo định dạng RosterViewer."""
    meta = os.path.join(OUT_DIR, "solutions", f"Instance{i}_{method}.json")
    best = float("inf")
    if os.path.exists(meta):
        with open(meta) as f:
            best = json.load(f)["cost"]
    if cost < best:
        with open(os.path.join(OUT_DIR, "solutions", f"Instance{i}_{method}.txt"), "w") as f:
            f.write(roster_to_text(inst, roster))
        with open(meta, "w") as f:
            json.dump(dict(cost=cost, roster=roster), f)

def run_all():
    done = load_results()
    traces = load_traces()
    for i in INSTANCES:
        inst = INST[i]
        print(f"\n=== Instance{i}: {inst.N} nhân viên, {inst.D} ngày, {inst.S} loại ca | BKS = {BKS[i]} ===")
        # ---- 1) Greedy
        if not already_done(done, i, "Greedy", 0, 0):
            t0 = time.perf_counter()
            g = greedy_construct(inst, random.Random(0))
            ev = evaluate(inst, g)
            append_row(dict(instance=i, method="Greedy", seed=0, time_limit=0,
                            cost=ev["soft"] if ev["feasible"] else np.nan, feasible=ev["feasible"],
                            hard_violations=ev["hard"], elapsed=time.perf_counter() - t0))
            print(f"  Greedy : vi phạm cứng = {ev['hard']}, điểm phạt mềm = {ev['soft']}")
        # ---- 2) MILP (HiGHS)
        if not already_done(done, i, "MILP", 0, MILP_TIME):
            m = solve_milp(inst, time_limit=MILP_TIME)
            hv = evaluate(inst, m["roster"])["hard"] if m["roster"] is not None else np.nan
            append_row(dict(instance=i, method="MILP", seed=0, time_limit=MILP_TIME, cost=m["cost"],
                            feasible=m["feasible"], hard_violations=hv, elapsed=m["elapsed"],
                            proven_optimal=m["proven_optimal"], lower_bound=m["lower_bound"]))
            if m["feasible"]:
                save_solution(i, "MILP", inst, m["roster"], m["cost"])
            print(f"  MILP   : cost = {m['cost']}, tối ưu = {m['proven_optimal']}, {m['elapsed']:.0f}s")
        # ---- 3) Hill Climbing và 4) Simulated Annealing
        for method in ("HC", "SA"):
            for seed in range(N_RUNS):
                if already_done(done, i, method, seed, TIME_LIMIT):
                    continue
                r = local_search(inst, TIME_LIMIT, seed=seed, method=method, **SA_PARAMS)
                ev = evaluate(inst, r["roster"])                 # kiểm tra lại bằng bộ độc lập
                assert ev["feasible"] == r["feasible"]
                if r["feasible"]:
                    assert ev["soft"] == r["cost"], "Chi phí SA báo cáo khác bộ kiểm tra độc lập"
                    save_solution(i, method, inst, r["roster"], r["cost"])
                append_row(dict(instance=i, method=method, seed=seed, time_limit=TIME_LIMIT,
                                cost=r["cost"], feasible=r["feasible"], hard_violations=ev["hard"],
                                time_to_best=r["time_to_best"], iterations=r["iterations"],
                                elapsed=r["elapsed"]))
                traces[f"{i}|{method}|{seed}|{TIME_LIMIT}"] = r["trace"]
                with open(TRACE_JSON, "w") as f:
                    json.dump(traces, f)
                gap = 100 * (r["cost"] - BKS[i]) / BKS[i] if r["feasible"] else float("nan")
                print(f"  {method} seed={seed}: cost = {r['cost']}, gap = {gap:.2f}%, "
                      f"{r['iterations']/1e6:.1f}M bước lặp")
    print("\nHoàn tất. Kết quả thô:", RESULT_CSV)

RUN_ORIGINAL = False   # True: chạy lại thực nghiệm gốc (~4 giờ). Bản chỉnh sửa: để False
if RUN_ORIGINAL:
    run_all()
else:
    print("Bỏ qua thực nghiệm gốc, dùng kết quả đã có:", RESULT_CSV, "| tồn tại:", os.path.exists(RESULT_CSV))


## 11. Bảng tổng hợp kết quả
`gap% = 100 × (chi phí − BKS) / BKS`. Trung bình và độ lệch chuẩn tính trên các lần chạy khả thi.

In [ ]:
def build_summary():
    df = load_results()
    df = df[df.instance.isin(INSTANCES)]
    rows = []
    for i in INSTANCES:
        b = BKS[i]
        row = dict(Instance=i, BKS=b)
        g = df[(df.instance == i) & (df.method == "Greedy")]
        if len(g):
            row["Greedy_hard_viol"] = int(g.hard_violations.iloc[0])
        m = df[(df.instance == i) & (df.method == "MILP") & (df.time_limit == MILP_TIME)]
        if len(m):
            row["MILP_cost"] = m.cost.iloc[0]
            row["MILP_gap%"] = 100 * (m.cost.iloc[0] - b) / b
            row["MILP_optimal"] = bool(m.proven_optimal.iloc[0])
            row["MILP_time_s"] = m.elapsed.iloc[0]
        for meth in ("HC", "SA"):
            r = df[(df.instance == i) & (df.method == meth) & (df.time_limit == TIME_LIMIT)]
            if not len(r):
                continue
            feas = r[r.feasible.astype(bool)]
            row[f"{meth}_feasible%"] = 100 * len(feas) / len(r)
            if len(feas):
                row[f"{meth}_best"] = feas.cost.min()
                row[f"{meth}_mean"] = feas.cost.mean()
                row[f"{meth}_std"] = feas.cost.std(ddof=1) if len(feas) > 1 else 0.0
                row[f"{meth}_gap_best%"] = 100 * (feas.cost.min() - b) / b
                row[f"{meth}_gap_mean%"] = 100 * (feas.cost.mean() - b) / b
                row[f"{meth}_time_to_best_s"] = feas.time_to_best.mean()
        rows.append(row)
    return pd.DataFrame(rows)

summary = build_summary()
summary.to_csv(os.path.join(OUT_DIR, "table_summary.csv"), index=False)
pd.set_option("display.max_columns", 50, "display.width", 250)
summary.round(2)

## 12. Kiểm định thống kê

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu

def paired_test(summary, col_a, col_b, label):
    """Wilcoxon signed-rank trên các instance mà cả hai phương pháp đều có lời giải khả thi."""
    if col_a not in summary or col_b not in summary:
        print(f"{label}: thiếu dữ liệu"); return
    s = summary[["Instance", col_a, col_b]].dropna()
    diff = s[col_a] - s[col_b]
    print(f"{label}: {len(s)} instance dùng được | A tốt hơn: {(diff < 0).sum()}, hòa: {(diff == 0).sum()}, "
          f"B tốt hơn: {(diff > 0).sum()}")
    if len(s) < 6 or (diff != 0).sum() == 0:
        print("   -> Chưa đủ cặp khác nhau để kiểm định có ý nghĩa (nên có >= 6 instance).")
        return
    stat, p = wilcoxon(s[col_a], s[col_b])
    print(f"   Wilcoxon signed-rank: W = {stat:.1f}, p = {p:.4f} "
          f"({'khác biệt có ý nghĩa ở mức 0.05' if p < 0.05 else 'chưa có ý nghĩa ở mức 0.05'})")

print("Kiểm định ghép cặp theo instance (A = SA):")
paired_test(summary, "SA_mean", "HC_mean", "SA (trung bình) so với HC (trung bình)")
paired_test(summary, "SA_best", "MILP_cost", "SA (tốt nhất) so với MILP")

print("\nMann-Whitney U trên từng instance (SA so với HC, các lần chạy khả thi):")
_df = load_results()
for i in INSTANCES:
    a = _df[(_df.instance == i) & (_df.method == "SA") & (_df.time_limit == TIME_LIMIT) & (_df.feasible.astype(bool))].cost
    b = _df[(_df.instance == i) & (_df.method == "HC") & (_df.time_limit == TIME_LIMIT) & (_df.feasible.astype(bool))].cost
    if len(a) >= 3 and len(b) >= 3:
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        print(f"  Instance{i}: SA n={len(a)}, HC n={len(b)}, U = {u:.1f}, p = {p:.4f}")
    else:
        print(f"  Instance{i}: không đủ lần chạy khả thi (SA {len(a)}, HC {len(b)})")

## 13. Hình vẽ cho bài báo (lưu PNG 300 dpi trong thư mục `figures`)

In [ ]:
FIG = os.path.join(OUT_DIR, "figures")
traces = load_traces()

def mean_curve(i, method, grid):
    """Đường hội tụ trung bình: chi phí khả thi tốt nhất theo thời gian, lấy trung bình qua các seed."""
    curves = []
    for seed in range(N_RUNS):
        tr = traces.get(f"{i}|{method}|{seed}|{TIME_LIMIT}")
        if not tr:
            continue
        ts = np.array([t for t, c in tr]); cs = np.array([np.nan if c is None else c for t, c in tr], float)
        idx = np.searchsorted(ts, grid, side="right") - 1
        curves.append(cs[idx])
    if not curves:
        return None
    arr = np.array(curves)
    ok = (~np.isnan(arr)).sum(axis=0) > 0
    out = np.full(len(grid), np.nan)
    out[ok] = np.nanmean(arr[:, ok], axis=0)
    return out

# Hình 1: đường hội tụ SA và HC
show = INSTANCES[-4:] if len(INSTANCES) >= 4 else INSTANCES
grid = np.linspace(0, TIME_LIMIT, 300)
fig, axes = plt.subplots(1, len(show), figsize=(4.2 * len(show), 3.4), squeeze=False)
for ax, i in zip(axes[0], show):
    for method, style in (("HC", "--"), ("SA", "-")):
        y = mean_curve(i, method, grid)
        if y is not None:
            ax.plot(grid, y, style, label=method)
    ax.axhline(BKS[i], color="gray", lw=0.8, ls=":", label="BKS")
    ax.set_title(f"Instance{i}"); ax.set_xlabel("Thời gian (s)"); ax.set_ylabel("Chi phí tốt nhất")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_convergence.png"), dpi=300); plt.show()

# Hình 2: gap trung bình so với BKS theo instance
cols = [c for c in ("MILP_gap%", "HC_gap_mean%", "SA_gap_mean%") if c in summary]
if cols:
    ax = summary.set_index("Instance")[cols].plot(kind="bar", figsize=(max(6, 0.8 * len(summary)), 3.6))
    ax.set_ylabel("Gap so với BKS (%)"); ax.set_xlabel("Instance")
    ax.legend([c.replace("_gap_mean%", " (trung bình)").replace("_gap%", "") for c in cols])
    plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_gap.png"), dpi=300); plt.show()

# Hình 3: phân bố gap của SA qua các lần chạy
_df = load_results()
_sa = _df[(_df.method == "SA") & (_df.time_limit == TIME_LIMIT) & (_df.feasible.astype(bool)) & (_df.instance.isin(INSTANCES))]
data = [(100 * (_sa[_sa.instance == i].cost - BKS[i]) / BKS[i]).values for i in INSTANCES]
if any(len(d) for d in data):
    plt.figure(figsize=(max(6, 0.8 * len(INSTANCES)), 3.4))
    plt.boxplot([d if len(d) else [np.nan] for d in data])
    plt.xticks(range(1, len(INSTANCES) + 1), INSTANCES)
    plt.xlabel("Instance"); plt.ylabel("Gap của SA so với BKS (%)")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_sa_boxplot.png"), dpi=300); plt.show()

# Hình 4: minh họa lịch trực tốt nhất của SA trên một instance
def plot_roster(i, method="SA"):
    meta = os.path.join(OUT_DIR, "solutions", f"Instance{i}_{method}.json")
    if not os.path.exists(meta):
        print("Chưa có lời giải khả thi cho", i, method); return
    with open(meta) as f:
        sol = json.load(f)
    inst = INST[i]
    R = np.array(sol["roster"])
    fig, ax = plt.subplots(figsize=(min(14, 0.35 * inst.D + 2), min(10, 0.3 * inst.N + 1.5)))
    cmap = plt.get_cmap("tab20", inst.S + 1)
    ax.imshow(R, aspect="auto", cmap=cmap, vmin=-0.5, vmax=inst.S + 0.5)
    names = [""] + inst.shift_ids
    if inst.D <= 28 and inst.N <= 40:
        for n in range(inst.N):
            for d in range(inst.D):
                ax.text(d, n, names[R[n, d]], ha="center", va="center", fontsize=7)
    for w in range(inst.D // 7):
        ax.axvspan(7 * w + 4.5, 7 * w + 6.5, color="k", alpha=0.08)
    ax.set_yticks(range(inst.N)); ax.set_yticklabels(inst.staff_ids, fontsize=7)
    ax.set_xlabel("Ngày (0 = thứ Hai; vùng xám = cuối tuần)"); ax.set_ylabel("Nhân viên")
    ax.set_title(f"Instance{i} - lịch tốt nhất của {method} (chi phí {sol['cost']:.0f}, BKS {BKS[i]})")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"fig_roster_Instance{i}.png"), dpi=300); plt.show()

plot_roster(INSTANCES[min(1, len(INSTANCES) - 1)])

## 14. Tải toàn bộ kết quả về máy

In [ ]:
import shutil
shutil.make_archive("nrp_sa_results", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("nrp_sa_results.zip")
except ImportError:
    print("Đã nén kết quả vào nrp_sa_results.zip")

---
# PHẦN BỔ SUNG THEO Ý KIẾN PHẢN BIỆN (vòng 1)

Các ô dưới đây **không đụng tới** `results_raw.csv` / `traces.json` của thực nghiệm gốc. Mọi kết quả mới được ghi vào thư mục con `revision/` của `OUT_DIR`, mỗi thí nghiệm một file riêng, và **chạy tiếp được** nếu Colab bị ngắt (lần chạy đã xong sẽ được bỏ qua).

**Cách chạy**
1. Ở ô cấu hình (mục 7) để `MODE = "paper"` và `USE_DRIVE = True`.
2. Chạy các ô 1 → 9 (định nghĩa hàm, tải dữ liệu, kiểm tra tính đúng đắn). **Không cần chạy lại mục 10** (thực nghiệm gốc).
3. Chạy lần lượt các ô R0 → R7 bên dưới. Mỗi ô in ra thời gian ước lượng; nếu Colab ngắt, chỉ cần chạy lại ô 1 → 9 rồi chạy lại ô đang dở.
4. Chạy ô R8 (tổng hợp) và R9 (nén kết quả để gửi lại).

| Ô | Nội dung | Thời gian tối đa |
|---|---|---|
| R0 | Thông tin máy, phiên bản thư viện, kích thước mô hình MILP | < 1 phút |
| R1 | HiGHS với cùng ngân sách 60 giây (12 tập) | ≈ 13 phút |
| R2 | Số người thiếu tối thiểu có thể đạt (MILP chỉ tối thiểu hóa thiếu người, 60 s/tập) | ≈ 13 phút |
| R3 | SA 60 s đối chứng trên cùng máy (tập 5–12 × 5 lần), có ghi chỉ số mở rộng | ≈ 40 phút |
| R4 | SA 600 s (tập 5–12 × 5 lần) | ≈ 6,7 giờ |
| R5 | Hill Climbing khởi động lại ngẫu nhiên (12 tập × 10 lần × 60 s) | ≈ 2 giờ |
| R6 | Độ nhạy tham số T0, Tf, λ (tập 3, 5, 9, 11 × 7 cấu hình × 5 lần × 60 s) | ≈ 2,3 giờ |
| R7 | Chỉ số mở rộng cho thực nghiệm gốc (tỷ lệ lần chạy khả thi theo thời gian, cơ cấu chi phí) | < 1 phút |

**Không chạy song song nhiều ô hoặc nhiều tab Colab**: nhiệt độ của SA giảm theo thời gian thực, nên chạy song song làm thuật toán chậm đi và kết quả không còn so sánh được.

In [ ]:
# R-COMMON: hàm dùng chung cho các thí nghiệm bổ sung
import platform, sys, subprocess
REV_DIR = os.path.join(OUT_DIR, "revision")
os.makedirs(REV_DIR, exist_ok=True)

def rev_path(name):
    return os.path.join(REV_DIR, name)

def rev_load(name):
    p = rev_path(name)
    return pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()

def rev_done(df, **key):
    if df.empty:
        return False
    m = np.ones(len(df), bool)
    for k, v in key.items():
        if k not in df:
            return False
        m &= (df[k].astype(str) == str(v)).values
    return bool(m.any())

def rev_append(name, row):
    p = rev_path(name)
    pd.DataFrame([row]).to_csv(p, mode="a", header=not os.path.exists(p), index=False)

def rev_save_trace(name, key, trace):
    p = rev_path(name)
    d = {}
    if os.path.exists(p):
        with open(p) as f:
            d = json.load(f)
    d[key] = trace
    with open(p, "w") as f:
        json.dump(d, f)

def coverage_breakdown(inst, roster):
    # Tách điểm phạt mềm thành (i) thiếu/thừa người so với nhu cầu và (ii) nguyện vọng cá nhân.
    R = np.asarray(roster)
    under = over = under_cells = over_cells = 0
    under_cost = over_cost = 0.0
    for d in range(inst.D):
        for s in range(1, inst.S + 1):
            x, r = int((R[:, d] == s).sum()), int(inst.cov_req[d, s])
            if x < r:
                under += r - x; under_cells += 1; under_cost += (r - x) * float(inst.cov_under[d, s])
            elif x > r:
                over += x - r; over_cells += 1; over_cost += (x - r) * float(inst.cov_over[d, s])
    ev = evaluate(inst, roster)
    return dict(required_total=int(inst.cov_req[:, 1:].sum()), under_staff=under, over_staff=over,
                under_cells=under_cells, over_cells=over_cells, under_cost=under_cost,
                over_cost=over_cost, cover_cost=under_cost + over_cost,
                pref_cost=float(ev["soft_requests"]), hard_violations=int(ev["hard"]),
                **{f"viol_{k}": int(v) for k, v in ev["detail"].items()})

def first_feasible_time(trace):
    for t, c in trace:
        if c is not None and not (isinstance(c, float) and math.isnan(c)):
            return float(t)
    return float("nan")

def timed_local_search(inst, time_limit, **kw):
    # Bọc local_search để đo cả thời gian khởi tạo (tiền xử lý + lời giải tham lam),
    # vốn nằm ngoài đồng hồ t0 bên trong local_search.
    t_start = time.perf_counter()
    r = local_search(inst, time_limit, **kw)
    r["total_wall"] = time.perf_counter() - t_start
    r["setup_time"] = r["total_wall"] - r["elapsed"]
    r["overrun"] = r["elapsed"] - time_limit
    return r

def run_record(i, inst, r, **extra):
    # Lưu lịch trực của TỪNG lần chạy để người đọc kiểm tra lại (theo yêu cầu của Ban biên tập).
    tag = "_".join(f"{k}{v}" for k, v in extra.items() if k in ("method", "config", "seed", "time_limit"))
    os.makedirs(rev_path("solutions_runs"), exist_ok=True)
    with open(rev_path(f"solutions_runs/Instance{i}_{tag}.json"), "w") as f:
        json.dump(dict(instance=i, **{k: str(v) for k, v in extra.items()}, feasible=bool(r["feasible"]),
                       cost=r["cost"] if r["feasible"] else None, roster=r["roster"]), f)
    ev = evaluate(inst, r["roster"])
    assert ev["feasible"] == r["feasible"]
    if r["feasible"]:
        assert ev["soft"] == r["cost"], "Chi phí báo cáo khác bộ kiểm tra độc lập"
    row = dict(instance=i, **extra, cost=r["cost"], feasible=r["feasible"],
               gap=100 * (r["cost"] - BKS[i]) / BKS[i] if r["feasible"] else np.nan,
               time_to_best=r["time_to_best"], first_feasible=first_feasible_time(r["trace"]),
               iterations=r["iterations"], accepted=r["accepted"], search_time=r["elapsed"],
               setup_time=r["setup_time"], total_wall=r["total_wall"], overrun=r["overrun"])
    row.update(coverage_breakdown(inst, r["roster"]))
    return row

# Các hàm đọc kết quả gốc (giống mục 10 và 13) được định nghĩa lại ở đây để KHÔNG phải chạy lại mục 10-13.
RESULT_CSV = os.path.join(OUT_DIR, "results_raw.csv")
TRACE_JSON = os.path.join(OUT_DIR, "traces.json")
FIG = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG, exist_ok=True)

def load_results():
    return pd.read_csv(RESULT_CSV) if os.path.exists(RESULT_CSV) else pd.DataFrame(
        columns=["instance", "method", "seed", "time_limit", "cost", "feasible"])

def load_traces():
    if os.path.exists(TRACE_JSON):
        with open(TRACE_JSON) as f:
            return json.load(f)
    return {}

def mean_curve_from(traces, i, method, grid, tl, seeds):
    curves = []
    for seed in seeds:
        tr = traces.get(f"{i}|{method}|{seed}|{tl}")
        if not tr:
            continue
        ts = np.array([t for t, c in tr]); cs = np.array([np.nan if c is None else c for t, c in tr], float)
        curves.append(cs[np.searchsorted(ts, grid, side="right") - 1])
    if not curves:
        return None
    arr = np.array(curves); ok = (~np.isnan(arr)).sum(axis=0) > 0
    out = np.full(len(grid), np.nan); out[ok] = np.nanmean(arr[:, ok], axis=0)
    return out

print("Thư mục kết quả bổ sung:", REV_DIR)
print("Kết quả gốc:", RESULT_CSV, "(có)" if os.path.exists(RESULT_CSV) else "(CHƯA CÓ - hãy chép results_raw.csv, traces.json, solutions/ vào OUT_DIR)")

### R0. Môi trường tính toán và kích thước mô hình quy hoạch nguyên

In [ ]:
import scipy, numpy
def cpu_model():
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

def highs_version():
    try:
        import highspy
        h = highspy.Highs()
        return f"{h.version()} (highspy)"
    except Exception:
        pass
    try:
        from scipy.optimize._highspy import _core
        h = _core._Highs()
        return f"{h.version()} (kèm SciPy)"
    except Exception:
        pass
    try:
        from scipy.optimize._highs import _highs_wrapper
        return "kèm SciPy (không đọc được số phiên bản)"
    except Exception:
        return "kèm SciPy"

try:
    mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
except Exception:
    mem_gb = float("nan")
ENV = dict(cpu=cpu_model(), logical_cpus=os.cpu_count(), ram_gb=round(mem_gb, 1),
           os=platform.platform(), python=sys.version.split()[0], numpy=numpy.__version__,
           scipy=scipy.__version__, pandas=pd.__version__, highs=highs_version(),
           note="SA/HC chạy đơn luồng (Python thuần). HiGHS gọi qua scipy.optimize.milp với thiết lập luồng mặc định.")
# Không ghi đè cấu hình của máy đã chạy thực nghiệm; máy hiện tại được ghi vào file riêng.
_env_file = "environment_current.json" if os.path.exists(rev_path("environment.json")) else "environment.json"
with open(rev_path(_env_file), "w") as f:
    json.dump(ENV, f, indent=2, ensure_ascii=False)
for k, v in ENV.items():
    print(f"{k:13s}: {v}")

# Kích thước mô hình MILP: dựng mô hình đúng như solve_milp nhưng dừng trước khi giải.
import scipy.optimize as _so
class _Captured(Exception):
    pass

def milp_model_size(inst):
    orig = _so.milp
    def stub(c, constraints=None, integrality=None, bounds=None, options=None):
        A = constraints.A
        n_bin = int(((bounds.ub == 1) | (bounds.ub == 0)).sum())
        raise _Captured(dict(variables=A.shape[1], binary=n_bin, integer=A.shape[1] - n_bin,
                             constraints=A.shape[0], nonzeros=int(A.nnz)))
    _so.milp = stub
    try:
        solve_milp(inst, 1)
    except _Captured as e:
        return e.args[0]
    finally:
        _so.milp = orig

size_rows = []
for i in INSTANCES:
    inst = INST[i]
    s = milp_model_size(inst)
    size_rows.append(dict(Instance=i, N=inst.N, D=inst.D, S=inst.S, x_vars=inst.N * inst.D * inst.S,
                          weekend_vars=inst.N * (inst.D // 7), slack_vars=2 * inst.D * inst.S, **s))
milp_size = pd.DataFrame(size_rows)
milp_size.to_csv(rev_path("milp_model_size.csv"), index=False)
milp_size

### R1. HiGHS với cùng ngân sách 60 giây
Thời gian `total_wall` gồm cả bước dựng mô hình; `solve_time` là thời gian bộ giải.

In [ ]:
HIGHS_BUDGETS = [60]
def run_highs_budget():
    done = rev_load("highs_budget.csv")
    for tl in HIGHS_BUDGETS:
        for i in INSTANCES:
            if rev_done(done, instance=i, time_limit=tl):
                continue
            inst = INST[i]
            t_start = time.perf_counter()
            m = solve_milp(inst, time_limit=tl)
            wall = time.perf_counter() - t_start
            row = dict(instance=i, time_limit=tl, status=m["status"], message=m["message"],
                       feasible=m["feasible"], cost=m["cost"],
                       gap=100 * (m["cost"] - BKS[i]) / BKS[i] if m["feasible"] else np.nan,
                       proven_optimal=m["proven_optimal"], lower_bound=m["lower_bound"],
                       solve_time=m["elapsed"], total_wall=wall)
            if m["roster"] is not None:
                row.update(coverage_breakdown(inst, m["roster"]))
                os.makedirs(rev_path("solutions_runs"), exist_ok=True)
                with open(rev_path(f"solutions_runs/Instance{i}_methodHiGHS_time_limit{tl}.json"), "w") as f:
                    json.dump(dict(instance=i, method="HiGHS", time_limit=tl, cost=m["cost"], roster=m["roster"]), f)
            rev_append("highs_budget.csv", row)
            print(f"Instance{i} | HiGHS {tl}s: cost = {m['cost']}, tối ưu = {m['proven_optimal']}, "
                  f"{m['elapsed']:.1f}s")
print(f"Ước lượng tối đa: {len(INSTANCES) * sum(HIGHS_BUDGETS) / 60:.0f} phút")
run_highs_budget()
rev_load("highs_budget.csv")[["instance", "time_limit", "feasible", "cost", "gap", "proven_optimal", "solve_time"]]

### R2. Mức thiếu người tối thiểu có thể đạt được
Giải lại mô hình MILP với mục tiêu **chỉ** là tổng số người thiếu (bỏ qua nguyện vọng và thừa người). Nếu giá trị tối ưu > 0 thì không tồn tại lịch nào vừa thỏa các ràng buộc cứng vừa đủ người ở mọi ca, tức là không thể chuyển nhu cầu nhân lực tối thiểu thành ràng buộc cứng trên bộ dữ liệu này.

In [ ]:
def solve_min_shortage(inst, time_limit=60):
    orig = _so.milp
    N, D, S = inst.N, inst.D, inst.S
    first_un = N * D * S + N * (D // 7)
    def patched(c, constraints=None, integrality=None, bounds=None, options=None):
        c2 = np.zeros_like(c)
        c2[first_un:first_un + D * S] = 1.0            # chỉ phạt biến thiếu người UN[d,s]
        return orig(c2, constraints=constraints, integrality=integrality, bounds=bounds, options=options)
    _so.milp = patched
    try:
        m = solve_milp(inst, time_limit=time_limit)
    finally:
        _so.milp = orig
    const = float(sum(inst.on_w[n, d, s] for n in range(N) for d in range(D) for s in range(1, S + 1)))
    lb = m["lower_bound"] - const if np.isfinite(m["lower_bound"]) else np.nan
    out = dict(status=m["status"], proven_optimal=m["proven_optimal"], solve_time=m["elapsed"],
               min_under_staff=np.nan, under_staff_lower_bound=math.ceil(lb - 1e-6) if np.isfinite(lb) else np.nan)
    if m["roster"] is not None:
        cb = coverage_breakdown(inst, m["roster"])
        out.update(min_under_staff=cb["under_staff"], hard_violations=cb["hard_violations"])
    return out

def run_min_shortage(tl=60):
    done = rev_load("min_shortage.csv")
    for i in INSTANCES:
        if rev_done(done, instance=i):
            continue
        r = solve_min_shortage(INST[i], tl)
        rev_append("min_shortage.csv", dict(instance=i, time_limit=tl,
                                            required_total=int(INST[i].cov_req[:, 1:].sum()), **r))
        print(f"Instance{i}: số người thiếu tối thiểu = {r['min_under_staff']} "
              f"(cận dưới = {r['under_staff_lower_bound']}, chứng minh tối ưu = {r['proven_optimal']}, {r['solve_time']:.1f}s)")
run_min_shortage()
rev_load("min_shortage.csv")

### R3 + R4. SA 60 giây (đối chứng trên cùng máy) và SA 600 giây
Chạy trên các tập 5–12 (các tập SA còn cách BKS nhiều), 5 lần mỗi tập, seed 0–4. SA 60 s được chạy lại trên **cùng máy** để so sánh công bằng với SA 600 s; kết quả gốc trong `results_raw.csv` không bị thay đổi.

In [ ]:
LONG_INSTANCES = [i for i in INSTANCES if i >= 5]
LONG_SEEDS = list(range(5))

def run_sa_budget(time_limit):
    done = rev_load("sa_budget.csv")
    todo = [(i, s) for i in LONG_INSTANCES for s in LONG_SEEDS
            if not rev_done(done, instance=i, method="SA", seed=s, time_limit=time_limit)]
    print(f"SA {time_limit}s: còn {len(todo)} lần chạy, tối đa {len(todo) * time_limit / 3600:.1f} giờ")
    for i, seed in todo:
        inst = INST[i]
        r = timed_local_search(inst, time_limit, seed=seed, method="SA", **SA_PARAMS)
        rev_append("sa_budget.csv", run_record(i, inst, r, method="SA", seed=seed, time_limit=time_limit))
        rev_save_trace("traces_sa_budget.json", f"{i}|SA|{seed}|{time_limit}", r["trace"])
        print(f"  Instance{i} seed={seed} {time_limit}s: cost = {r['cost']}, "
              f"vượt giờ = {r['overrun']:.3f}s, khởi tạo = {r['setup_time']:.2f}s")

run_sa_budget(60)     # R3: khoảng 40 phút

In [ ]:
run_sa_budget(600)    # R4: khoảng 6,7 giờ, chạy tiếp được nếu Colab ngắt

### R5. Hill Climbing khởi động lại ngẫu nhiên (RR-HC)
Cùng ngân sách 60 giây, cùng ba phép biến đổi lân cận và hàm chi phí như SA. Mỗi lần khởi động lại dùng một lời giải tham lam được ngẫu nhiên hóa (chọn ngẫu nhiên trong nhóm ứng viên tốt nhất, kiểu GRASP) rồi leo đồi; khởi động lại khi lời giải tốt nhất của lượt hiện tại không cải thiện trong `RR_STALL` giây. Ngân sách 60 giây tính **cả** thời gian dựng lời giải khởi tạo.

In [ ]:
RR_STALL = 5.0       # giây không cải thiện thì khởi động lại
RR_ALPHA = 0.5       # độ rộng nhóm ứng viên: chọn trong ceil(need * (1 + alpha)) + 1 ứng viên tốt nhất

def randomized_greedy(inst, rng, alpha=RR_ALPHA):
    # Giống greedy_construct nhưng chọn ngẫu nhiên trong nhóm ứng viên tốt nhất (restricted candidate list).
    N, D, S = inst.N, inst.D, inst.S
    roster = [[0] * D for _ in range(N)]
    minutes = [0] * N; cnt = [[0] * (S + 1) for _ in range(N)]
    run = [0] * N; offrun = [99] * N; wk_used = [0] * N
    for d in range(D):
        is_we = (d % 7) >= 5
        sat_worked = [(d % 7 == 6) and roster[n][d - 1] > 0 for n in range(N)]
        order = sorted(range(1, S + 1), key=lambda s: (-int(inst.cov_req[d, s] * inst.cov_under[d, s]), rng.random()))
        for s in order:
            need = int(inst.cov_req[d, s])
            cands = []
            for n in range(N):
                if roster[n][d] or d in inst.days_off[n] or cnt[n][s] >= inst.max_shifts[n][s]:
                    continue
                if minutes[n] + inst.length[s] > inst.max_min[n] or run[n] >= inst.max_cons[n]:
                    continue
                if d > 0 and inst.forbid[roster[n][d - 1]][s]:
                    continue
                if is_we and not sat_worked[n] and wk_used[n] >= inst.max_wk[n]:
                    continue
                continuing = 0 < run[n] < inst.min_cons[n]
                rested = run[n] > 0 or offrun[n] >= inst.min_off[n]
                pref = int(inst.on_w[n, d, s]) - int(inst.off_w[n, d, s])
                cands.append((-int(continuing), -int(rested), -pref, -(inst.min_min[n] - minutes[n]), rng.random(), n))
            cands.sort()
            rcl = cands[:int(math.ceil(need * (1 + alpha))) + 1]
            for c in rng.sample(rcl, min(need, len(rcl))):
                n = c[-1]
                roster[n][d] = s; minutes[n] += inst.length[s]; cnt[n][s] += 1
                if is_we and not sat_worked[n]:
                    wk_used[n] += 1
        for n in range(N):
            if roster[n][d]:
                run[n] += 1; offrun[n] = 0
            else:
                run[n] = 0; offrun[n] += 1
    return roster

def rr_hill_climbing(inst, time_limit, seed=0, stall=RR_STALL):
    # Lặp: khởi tạo ngẫu nhiên hóa -> HC theo từng lát thời gian ngắn; dừng lượt khi không cải thiện
    # trong `stall` giây; giữ lời giải khả thi tốt nhất qua mọi lượt. Ngân sách tính cả khởi tạo.
    rng = random.Random(10_000 + seed)
    t_start = time.perf_counter()
    best_cost, best_roster, t_best, best_any = None, None, 0.0, None
    trace, restarts, iters = [(0.0, float("nan"))], 0, 0
    while True:
        left = time_limit - (time.perf_counter() - t_start)
        if left <= 0.05:
            break
        restarts += 1
        cur = randomized_greedy(inst, rng)
        cur_cost, cur_feas, last_imp = None, False, time.perf_counter()
        while True:
            left = time_limit - (time.perf_counter() - t_start)
            if left <= 0.05 or time.perf_counter() - last_imp >= stall:
                break
            sl = min(1.0, left)
            r = local_search(inst, sl, seed=rng.randrange(1 << 30), method="HC",
                             p_moves=SA_PARAMS["p_moves"], max_block=SA_PARAMS["max_block"],
                             hard_weight=SA_PARAMS["hard_weight"], init=cur)
            iters += r["iterations"]
            ev = evaluate(inst, r["roster"])
            score = (0 if ev["feasible"] else 1, ev["hard"], ev["soft"])
            if cur_cost is None or score < cur_cost:
                cur_cost, cur, last_imp = score, r["roster"], time.perf_counter()
            if ev["feasible"] and (best_cost is None or ev["soft"] < best_cost):
                best_cost, best_roster = ev["soft"], [row[:] for row in r["roster"]]
                t_best = time.perf_counter() - t_start
                trace.append((t_best, float(best_cost)))
            if best_any is None or score < best_any[0]:
                best_any = (score, [row[:] for row in r["roster"]])
    total = time.perf_counter() - t_start
    final = best_roster if best_roster is not None else best_any[1]
    return dict(roster=final, feasible=best_roster is not None,
                cost=float(best_cost) if best_cost is not None else float("nan"),
                time_to_best=t_best, iterations=iters, accepted=np.nan, elapsed=total, trace=trace,
                total_wall=total, setup_time=0.0, overrun=total - time_limit, restarts=restarts)

def run_rrhc(n_runs=N_RUNS, time_limit=TIME_LIMIT):
    done = rev_load("rrhc.csv")
    todo = [(i, s) for i in INSTANCES for s in range(n_runs)
            if not rev_done(done, instance=i, method="RRHC", seed=s, time_limit=time_limit)]
    print(f"RR-HC: còn {len(todo)} lần chạy, tối đa {len(todo) * time_limit / 3600:.1f} giờ")
    for i, seed in todo:
        inst = INST[i]
        r = rr_hill_climbing(inst, time_limit, seed=seed)
        row = run_record(i, inst, r, method="RRHC", seed=seed, time_limit=time_limit)
        row["restarts"] = r["restarts"]
        rev_append("rrhc.csv", row)
        rev_save_trace("traces_rrhc.json", f"{i}|RRHC|{seed}|{time_limit}", r["trace"])
        print(f"  Instance{i} seed={seed}: cost = {r['cost']}, {r['restarts']} lần khởi động")
run_rrhc()

### R6. Độ nhạy tham số T0, Tf, λ
Thay đổi **từng tham số một** quanh cấu hình mặc định (T0 = w_max, Tf = 0,3, λ = 2·w_max). Dùng các tập 3, 5, 9, 11, **không** thuộc hai tập dùng để chọn tham số (Instance 1 và 2).

In [ ]:
SENS_INSTANCES = [i for i in (3, 5, 9, 11) if i in INSTANCES]
SENS_SEEDS = list(range(5))
SENS_CONFIGS = {             # (hệ số T0 theo w_max, Tf, hệ số λ theo w_max)
    "base":     (1.0, 0.3, 2.0),
    "T0x0.5":   (0.5, 0.3, 2.0),
    "T0x2":     (2.0, 0.3, 2.0),
    "Tf=0.1":   (1.0, 0.1, 2.0),
    "Tf=1.0":   (1.0, 1.0, 2.0),
    "lambda=1": (1.0, 0.3, 1.0),
    "lambda=4": (1.0, 0.3, 4.0),
}

def run_sensitivity(time_limit=TIME_LIMIT):
    done = rev_load("sensitivity.csv")
    todo = [(c, i, s) for c in SENS_CONFIGS for i in SENS_INSTANCES for s in SENS_SEEDS
            if not rev_done(done, config=c, instance=i, seed=s)]
    print(f"Độ nhạy: còn {len(todo)} lần chạy, tối đa {len(todo) * time_limit / 3600:.1f} giờ")
    for c, i, seed in todo:
        inst = INST[i]
        w = float(max(1, int(inst.cov_under.max())))
        a, tf, b = SENS_CONFIGS[c]
        p = dict(SA_PARAMS, T0=a * w, Tf=tf, hard_weight=b * w)
        r = timed_local_search(inst, time_limit, seed=seed, method="SA", **p)
        rev_append("sensitivity.csv", run_record(i, inst, r, method="SA", config=c, seed=seed,
                                                 time_limit=time_limit, T0=p["T0"], Tf=tf,
                                                 hard_weight=p["hard_weight"]))
        print(f"  {c:9s} Instance{i} seed={seed}: cost = {r['cost']}")
run_sensitivity()

### R7. Chỉ số mở rộng cho thực nghiệm gốc (60 giây × 10 lần)
* Tỷ lệ lần chạy đã có lịch khả thi theo thời gian (đọc từ `traces.json` gốc).
* Cơ cấu chi phí của lịch tốt nhất đã lưu: thiếu/thừa người và nguyện vọng.

In [ ]:
traces0 = load_traces()
grid = np.linspace(0, TIME_LIMIT, 241)
feas_rows = []
for i in INSTANCES:
    for method in ("HC", "SA"):
        ff = []
        for seed in range(N_RUNS):
            tr = traces0.get(f"{i}|{method}|{seed}|{TIME_LIMIT}")
            if tr is not None:
                ff.append(first_feasible_time(tr))
        if not ff:
            continue
        ff = np.array(ff, float)
        feas_rows.append(dict(instance=i, method=method, runs=len(ff),
                              feasible_runs=int(np.isfinite(ff).sum()),
                              first_feasible_median=float(np.nanmedian(ff)) if np.isfinite(ff).any() else np.nan,
                              first_feasible_max=float(np.nanmax(ff)) if np.isfinite(ff).any() else np.nan))
feas_time = pd.DataFrame(feas_rows)
feas_time.to_csv(rev_path("first_feasible_original.csv"), index=False)

def feasible_share(i, method, traces, tl, seeds):
    ff = [first_feasible_time(traces[k]) for k in (f"{i}|{method}|{s}|{tl}" for s in seeds) if k in traces]
    ff = np.array(ff, float)
    return None if not len(ff) else np.array([(ff <= t).mean() for t in np.linspace(0, tl, 241)])

if not traces0:
    print("Không có traces.json của thực nghiệm gốc -> bỏ qua hình này (xem hình R10 từ dữ liệu R3-R4).")
show = [i for i in (6, 8, 10, 12) if i in INSTANCES] or INSTANCES[-4:]
if not traces0:
    print("Không có traces.json gốc -> bỏ qua hình của thực nghiệm gốc, xem hình từ các lần chạy mới bên dưới.")
    show = []
if traces0:
    fig, axes = plt.subplots(2, len(show), figsize=(4.2 * len(show), 6.2), squeeze=False, sharex=True)
    for k, i in enumerate(show):
        ax, ax2 = axes[0][k], axes[1][k]
        for method, style in (("HC", "--"), ("SA", "-")):
            y = mean_curve_from(traces0, i, method, grid, TIME_LIMIT, range(N_RUNS))
            if y is not None:
                ax.plot(grid, y, style, label=method)
            sh = feasible_share(i, method, traces0, TIME_LIMIT, range(N_RUNS))
            if sh is not None:
                ax2.step(grid, 100 * sh, style, where="post", label=method)
        ax.axhline(BKS[i], color="gray", lw=0.8, ls=":", label="BKS")
        ax.set_title(f"Instance{i}"); ax.set_ylabel("Chi phí tốt nhất (TB các lần khả thi)"); ax.legend(fontsize=8)
        ax2.set_ylim(-5, 105); ax2.set_xlabel("Thời gian (s)"); ax2.set_ylabel("% lần chạy đã khả thi")
    if show:
        plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_convergence_feasible_share.png"), dpi=300); plt.show()
    else:
        plt.close(fig)

    # Hình tương tự từ các lần chạy MỚI (R3, R4, R5): đường chi phí và % lần chạy đã khả thi theo thời gian.
    def _load_json(name):
        pth = rev_path(name)
        return json.load(open(pth)) if os.path.exists(pth) else {}
    tr_sa, tr_rr = _load_json("traces_sa_budget.json"), _load_json("traces_rrhc.json")
    series = [("SA 60 s", tr_sa, "SA", 60, LONG_SEEDS, "-"), ("SA 600 s", tr_sa, "SA", 600, LONG_SEEDS, "-."),
              ("RR-HC 60 s", tr_rr, "RRHC", TIME_LIMIT, range(N_RUNS), "--")]
    show2 = [i for i in (6, 8, 10, 12) if i in LONG_INSTANCES]
    if tr_sa and show2:
        fig, axes = plt.subplots(2, len(show2), figsize=(4.2 * len(show2), 6.2), squeeze=False)
        for k, i in enumerate(show2):
            ax, ax2 = axes[0][k], axes[1][k]
            for lab, trs, meth, tl, seeds, style in series:
                g2 = np.linspace(0, tl, 241)
                y = mean_curve_from(trs, i, meth, g2, tl, seeds)
                if y is not None:
                    ax.plot(g2, y, style, label=lab)
                sh = feasible_share(i, meth, trs, tl, seeds)
                if sh is not None:
                    ax2.step(g2, 100 * sh, style, where="post", label=lab)
            ax.axhline(BKS[i], color="gray", lw=0.8, ls=":", label="BKS")
            ax.set_xscale("symlog", linthresh=1); ax2.set_xscale("symlog", linthresh=1)
            ax.set_title(f"Instance{i}"); ax.set_ylabel("Chi phí tốt nhất (TB các lần khả thi)"); ax.legend(fontsize=7)
            ax2.set_ylim(-5, 105); ax2.set_xlabel("Thời gian (s, thang log)"); ax2.set_ylabel("% lần chạy đã khả thi")
        plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_convergence_feasible_share_new.png"), dpi=300); plt.show()


cov_rows = []
for i in INSTANCES:
    for method in ("SA", "HC", "MILP"):
        meta = os.path.join(OUT_DIR, "solutions", f"Instance{i}_{method}.json")
        if os.path.exists(meta):
            with open(meta) as f:
                sol = json.load(f)
            cov_rows.append(dict(instance=i, method=method, cost=sol["cost"],
                                 **coverage_breakdown(INST[i], sol["roster"])))
cov_best = pd.DataFrame(cov_rows)
cov_best.to_csv(rev_path("coverage_best_solutions.csv"), index=False)
display(feas_time)
cov_best[["instance", "method", "cost", "required_total", "under_staff", "over_staff",
          "under_cost", "over_cost", "pref_cost"]]

### R8. Tổng hợp và kiểm định thống kê cho bản giải trình

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for r, k in enumerate(order):
        run = max(run, (m - r) * p[k]); adj[k] = min(1.0, run)
    return adj

def compare_runs(A, B, name_a, name_b, instances, label):
    # Mann-Whitney U chính xác (hai phía) trên các lần chạy khả thi, hiệu chỉnh Holm; hiệu ứng rank-biserial.
    rows = []
    for i in instances:
        a = A[(A.instance == i) & A.feasible.astype(bool)].cost.values
        b = B[(B.instance == i) & B.feasible.astype(bool)].cost.values
        row = dict(instance=i, **{f"{name_a}_feasible": f"{len(a)}/{(A.instance == i).sum()}",
                                  f"{name_b}_feasible": f"{len(b)}/{(B.instance == i).sum()}",
                                  f"{name_a}_mean": a.mean() if len(a) else np.nan,
                                  f"{name_b}_mean": b.mean() if len(b) else np.nan})
        if len(a) >= 3 and len(b) >= 3:
            u, p = mannwhitneyu(a, b, alternative="two-sided", method="exact")
            row.update(U=u, p=p, rank_biserial=1 - 2 * u / (len(a) * len(b)))
        rows.append(row)
    t = pd.DataFrame(rows)
    if "p" in t and t.p.notna().any():
        ok = t.p.notna()
        t.loc[ok, "p_holm"] = holm(t.loc[ok, "p"].values)
    print(f"\n{label}"); display(t.round(4))
    return t

orig = load_results()
SA60 = orig[(orig.method == "SA") & (orig.time_limit == TIME_LIMIT)]
HC60 = orig[(orig.method == "HC") & (orig.time_limit == TIME_LIMIT)]
out = {}

if not SA60.empty and not HC60.empty:
    out["SA_vs_HC"] = compare_runs(SA60, HC60, "SA", "HC", INSTANCES, "SA so với HC (thực nghiệm gốc)")
rr = rev_load("rrhc.csv")
if not rr.empty and not SA60.empty:
    out["SA_vs_RRHC"] = compare_runs(SA60, rr, "SA", "RRHC", INSTANCES, "SA so với HC khởi động lại ngẫu nhiên")

hb = rev_load("highs_budget.csv")
if not hb.empty:
    m60 = hb[hb.time_limit == 60].set_index("instance")
    t = []
    for i in INSTANCES:
        sa = SA60[(SA60.instance == i) & SA60.feasible.astype(bool)].cost
        t.append(dict(instance=i, SA60_feasible=f"{len(sa)}/{(SA60.instance == i).sum()}",
                      SA60_best=sa.min() if len(sa) else np.nan, SA60_mean=sa.mean() if len(sa) else np.nan,
                      HiGHS60_cost=m60.cost.get(i, np.nan), HiGHS60_optimal=m60.proven_optimal.get(i, np.nan),
                      HiGHS60_time=m60.solve_time.get(i, np.nan), BKS=BKS[i]))
    out["SA_vs_HiGHS60"] = pd.DataFrame(t)
    print("\nSA 60 s so với HiGHS 60 s"); display(out["SA_vs_HiGHS60"].round(2))

sb = rev_load("sa_budget.csv")
if not sb.empty:
    g = sb.groupby(["instance", "time_limit"]).agg(runs=("seed", "count"), feasible=("feasible", "sum"),
                                                   mean_cost=("cost", "mean"), best_cost=("cost", "min"),
                                                   mean_gap=("gap", "mean"), max_overrun=("overrun", "max"),
                                                   mean_setup=("setup_time", "mean")).reset_index()
    out["SA_60_vs_600"] = g
    print("\nSA 60 s và 600 s (cùng máy)"); display(g.round(3))
    w = g.pivot(index="instance", columns="time_limit", values="mean_cost").dropna()
    if 60 in w and 600 in w and len(w) >= 6:
        print("Wilcoxon (trung bình theo tập, 600 s so với 60 s):", wilcoxon(w[600], w[60], method="exact"))

se = rev_load("sensitivity.csv")
if not se.empty:
    tab = se.groupby(["config", "instance"]).agg(feasible=("feasible", "sum"), runs=("seed", "count"),
                                                 mean_gap=("gap", "mean")).reset_index()
    tab["cell"] = tab.apply(lambda r: f"{r.mean_gap:.2f} ({int(r.feasible)}/{int(r.runs)})", axis=1)
    out["sensitivity"] = tab.pivot(index="config", columns="instance", values="cell").reindex(list(SENS_CONFIGS))
    print("\nĐộ nhạy tham số: gap trung bình % (số lần khả thi)"); display(out["sensitivity"])

with pd.ExcelWriter(rev_path("revision_summary.xlsx")) as xw:
    for k, v in out.items():
        v.to_excel(xw, sheet_name=k[:31], index=(k == "sensitivity"))
print("\nĐã lưu", rev_path("revision_summary.xlsx"))

### R10. Hội tụ của SA 60 s và 600 s, kèm tỷ lệ lần chạy đã có lịch khả thi
Trục hoành là % ngân sách thời gian vì lịch nhiệt độ của SA co giãn theo ngân sách.

In [ ]:
# R10: Hình hội tụ của SA 60 s và 600 s kèm tỷ lệ lần chạy đã khả thi (dữ liệu R3-R4, cùng máy)
import json, os, math
import numpy as np, matplotlib.pyplot as plt
with open(rev_path("traces_sa_budget.json")) as f:
    trB = json.load(f)
BKS_ = BKS
def _ff(tr):
    for t, c in tr:
        if c is not None and not (isinstance(c, float) and math.isnan(c)):
            return float(t)
    return float("inf")
def _curves(i, tl, grid):
    cs, ffs = [], []
    for s in range(5):
        tr = trB.get(f"{i}|SA|{s}|{tl}")
        if not tr:
            continue
        ts = np.array([t for t, c in tr]); v = np.array([np.nan if c is None else c for t, c in tr], float)
        cs.append(v[np.searchsorted(ts, grid, side="right") - 1]); ffs.append(_ff(tr))
    if not cs:
        return None, None
    arr = np.array(cs); ok = (~np.isnan(arr)).sum(0) > 0
    mean = np.full(len(grid), np.nan); mean[ok] = np.nanmean(arr[:, ok], axis=0)
    share = np.array([(np.array(ffs) <= t).mean() for t in grid]) * 100
    return mean, share
show = [6, 8, 10, 12]
fig, axes = plt.subplots(2, len(show), figsize=(4.2 * len(show), 6.0), squeeze=False, sharex="col")
for k, i in enumerate(show):
    ax, ax2 = axes[0][k], axes[1][k]
    for tl, style, lab in ((60, "--", "SA 60 s"), (600, "-", "SA 600 s")):
        g = np.linspace(0, tl, 601)
        m, sh = _curves(i, tl, g)
        if m is not None:
            x = 100 * g / tl     # trục hoành: % ngân sách thời gian (lịch nhiệt độ của SA co giãn theo ngân sách)
            ax.plot(x, m, style, label=lab); ax2.step(x, sh, style, where="post", label=lab)
    ax.axhline(BKS_[i], color="gray", lw=0.8, ls=":", label="BKS")
    ax2.set_xlim(0, 100)
    ax.set_title(f"Instance {i}"); ax.legend(fontsize=8)
    ax2.set_ylim(-5, 105); ax2.set_xlabel("% ngân sách thời gian đã dùng")
    if k == 0:
        ax.set_ylabel("Chi phí tốt nhất\n(TB các lần đã khả thi)"); ax2.set_ylabel("% lần chạy đã có lịch khả thi")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_sa_60_vs_600.png"), dpi=300); plt.show()


### R9. Nén thư mục `revision/` để gửi lại

In [ ]:
import shutil
shutil.make_archive("nrp_revision_results", "zip", REV_DIR)
try:
    from google.colab import files
    files.download("nrp_revision_results.zip")
except ImportError:
    print("Đã nén vào nrp_revision_results.zip")